# TP: Vision Transformers and CLIP

**From image patches to a multimodal embedding space.**

In this practical session we open up a Vision Transformer (ViT) and follow an image through it, then we use the same
network as the vision tower of **CLIP** to connect images and text.

| Part | Topic | What you will do |
|---|---|---|
| 1 | Reading the architecture | identify the two towers of CLIP, patch size, depth, width, number of heads |
| 2 | The ViT pipeline, step by step | patchify, patch embedding, [CLS] token, positional embeddings, forward pass module by module |
| 3 | Inside multi-head attention | attention computed from the raw weights, attention maps head by head |
| 4 | CLIP: zero-shot classification | image and text embeddings, cosine similarity, prompts |
| 5 | What does CLIP look at? | test-time registers, text-conditioned attention maps |

Questions are numbered **Q\<part\>.\<n\>**. All the code is given and runs as is: the goal is to *read* a Transformer
and to experiment with it, not to rewrite one.

The model used throughout is an **OpenCLIP ViT-B/16** (vision tower: ViT-Base, 16x16 patches), packaged with
*test-time registers* by Jiang, Dravid et al. It runs on CPU in a few seconds per image; a GPU runtime is not required.

**Credits.**
- Vision Transformer: Dosovitskiy et al., *An Image is Worth 16x16 Words*, ICLR 2021. https://arxiv.org/abs/2010.11929
- CLIP: Radford et al., *Learning Transferable Visual Models From Natural Language Supervision*, ICML 2021. https://arxiv.org/abs/2103.00020
- Registers: Darcet et al., *Vision Transformers Need Registers*, ICLR 2024. https://arxiv.org/abs/2309.16588
- Test-time registers and model weights: Jiang, Dravid, Efros, Gandelsman, *Vision Transformers Don't Need Trained Registers*, NeurIPS 2025. https://github.com/nickjiang2378/test-time-registers
- Parts 2 and 3 are adapted from the ViT/timm practical by Matteo Bastico (Mines Paris) and the tutorial by Hiroto Honda.

## 0. Setup

Run the two cells below once. On Google Colab the first cell installs the dependencies and fetches the images.

In [ ]:
# Dependencies. The model code on the Hugging Face Hub only loads its weights correctly with transformers <= 4.48
# (newer versions silently leave the weights uninitialised), hence the pin.
%pip install -q "transformers>=4.45,<4.49" ftfy einops regex

# Images (downloaded from the course repository if not already present next to the notebook)
import os, urllib.request
os.makedirs("images", exist_ok=True)
for name in ["shark.JPEG", "bird_and_shark.jpg", "scene.jpg"]:
    if not os.path.exists(f"images/{name}"):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/Rxzh/tp-vit-clip/main/images/{name}", f"images/{name}")
print("Images available:", os.listdir("images"))

In [ ]:
import math
import inspect
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

## 1. Reading the architecture

We load the model from the Hugging Face Hub. `trust_remote_code=True` is needed because the model comes with its own
Python code (a modified OpenCLIP).

In [ ]:
model = AutoModel.from_pretrained("amildravid4292/clip-vitb16-test-time-registers", trust_remote_code=True)
model = model.to(device).eval()
clip = model.model          # the CLIP network itself
vit = clip.visual           # its vision tower (a ViT-B/16)

# Sanity check: with an incompatible transformers version the weights are silently left uninitialised.
import transformers
assert not any(torch.isnan(p).any() for p in model.parameters()) and clip.logit_scale.item() > 1, \
    f"weights not loaded (transformers {transformers.__version__}); run the setup cell and restart the runtime"
print("weights loaded, transformers", transformers.__version__)

### 1.1 The two towers of CLIP

In [ ]:
print(clip)

**Q1.1** CLIP is made of two encoders ("towers"). Identify them in the printed module. What is the input of each one,
and what do they have in common at the output?

### 1.2 The vision tower

In [ ]:
print(vit)

In [ ]:
attn0 = vit.transformer.resblocks[0].attn
print("in_proj_weight:", tuple(attn0.in_proj_weight.shape))
print("head_dim      :", attn0.head_dim)

**Q1.2** From the printed vision tower and the two shapes above, give:

1. the patch size;
2. the number of Transformer layers;
3. the embedding dimension (width) of the tokens;
4. the number of attention heads, knowing that each head works in dimension `head_dim = 64`.

Explain where the `2304` of `in_proj_weight` comes from (it is **not** the number of heads).

In [ ]:
# Check your answers
patch_size = vit.patch_size[0]
num_layers = len(vit.transformer.resblocks)
width = vit.positional_embedding.shape[1]
num_heads = attn0.num_heads
print(f"patch_size={patch_size}, num_layers={num_layers}, width={width}, num_heads={num_heads}, head_dim={attn0.head_dim}")

In [ ]:
print("positional_embedding:", tuple(vit.positional_embedding.shape))

**Q1.3** Why does the positional embedding contain 197 vectors for a 224x224 image?

## 2. The ViT pipeline, step by step

The ViT pipeline is:

1. **Split the image into patches** and embed each patch as a vector (patch embedding).
2. **Prepend a [CLS] token** and **add positional embeddings**.
3. **Transformer encoder**: 12 residual blocks (multi-head attention + MLP), the sequence keeps its shape.
4. **Read out** the [CLS] token, which summarises the image.

Let us follow one image through these steps.

In [ ]:
im = Image.open("images/shark.JPEG").convert("RGB").resize((224, 224))
plt.imshow(im); plt.title("Input"); plt.axis("off"); plt.show()

# CLIP's own preprocessing: resize, center-crop, to tensor, normalise
img = model.preprocessor(im).unsqueeze(0).to(device)    # [1, 3, 224, 224]
print("image tensor:", tuple(img.shape))

### 2.1 Patchify

In [ ]:
def img_to_patch(x, patch_size, flatten_channels=True):
    '''
    x            - tensor of shape [B, C, H, W]
    patch_size   - number of pixels per side of a patch
    returns      - [B, N, C*p*p] if flatten_channels else [B, N, C, p, p], with N = (H/p)*(W/p)
    '''
    B, C, H, W = x.shape
    x = x.reshape(B, C, H // patch_size, patch_size, W // patch_size, patch_size)
    x = x.permute(0, 2, 4, 1, 3, 5)     # [B, H', W', C, p, p]
    x = x.flatten(1, 2)                 # [B, H'*W', C, p, p]
    if flatten_channels:
        x = x.flatten(2, 4)             # [B, H'*W', C*p*p]
    return x

img_patches = img_to_patch(TF.to_tensor(im).unsqueeze(0), patch_size, flatten_channels=False)   # [1, 196, 3, 16, 16]
grid = torchvision.utils.make_grid(img_patches[0], nrow=14, normalize=True, pad_value=0.9)
plt.figure(figsize=(5, 5)); plt.imshow(grid.permute(1, 2, 0)); plt.axis("off")
plt.title("196 patches of 16x16 pixels (14 x 14 grid)"); plt.show()

### 2.2 Patch embedding

Each 16x16x3 patch (768 pixel values) must become a 768-dimensional token. The ViT does it with the `conv1` layer.

In [ ]:
with torch.no_grad():
    conv_out = vit.conv1(img)
print("conv1 output:", tuple(conv_out.shape))

**Q2.1** How does a convolution with `kernel_size = stride = 16` and no padding implement "split into patches, then
apply the same linear layer to every patch"? Why does the output have shape `[1, 768, 14, 14]`?

Let us verify this claim: we compute the patch embedding **without** the convolution, with `img_to_patch` and a
matrix product with the convolution weights, and check that we recover `conv_out`.

In [ ]:
W = vit.conv1.weight                      # [768, 3, 16, 16]
flat_patches = img_to_patch(img, patch_size)      # [1, 196, 768]

W_mat = W.reshape(W.shape[0], -1)                 # [768, 3*16*16] = [768, 768]
tokens_manual = flat_patches @ W_mat.T            # [1, 196, 768]

tokens_conv = conv_out.flatten(2).transpose(1, 2)   # [1, 768, 14, 14] -> [1, 196, 768]
print("same result:", torch.allclose(tokens_manual, tokens_conv, atol=1e-4))

### 2.3 [CLS] token and positional embeddings

In [ ]:
print("class_embedding     :", tuple(vit.class_embedding.shape))
print("positional_embedding:", tuple(vit.positional_embedding.shape))

**Q2.2** What is the role of the [CLS] token? Where does it come from, and where is it read at the end?

**Q2.3** Self-attention treats its input as a *set*: permuting the tokens permutes the outputs in the same way
(permutation equivariance). Why is this a problem for images, and how do positional embeddings fix it?

Here is the full forward pass of the vision tower **written by hand**, module by module. Read it alongside the
printed architecture of Q1.2: the shapes are given in the comments, and the result is compared with `model.encode_image`.

In [ ]:
with torch.no_grad():
    y = vit.patchnorm_pre_ln(img)                          # [1, 3, 224, 224]   (identity here)
    y = vit.conv1(y)                                       # [1, 768, 14, 14]   patch embedding
    y = y.reshape(y.shape[0], y.shape[1], -1).permute(0, 2, 1)   # [1, 196, 768]   grid -> sequence

    cls = vit.class_embedding.to(y.dtype).reshape(1, 1, -1).expand(y.shape[0], -1, -1)   # [1, 1, 768]
    y = torch.cat([cls, y], dim=1)                          # [1, 197, 768]

    y = y + vit.positional_embedding.to(y.dtype)           # [1, 197, 768]

    y = vit.ln_pre(y)                                      # [1, 197, 768]   layer norm
    y = vit.transformer(y)                                 # [1, 197, 768]   12 residual blocks
    y = vit.ln_post(y)                                     # [1, 197, 768]

    y = y[:, 0, :]                                         # [1, 768]        [CLS] token only
    image_features_manual = y @ vit.proj                   # [1, 512]

    image_features = model.encode_image(img, num_register_tokens=0)
print("manual :", tuple(image_features_manual.shape))
print("encode_image:", tuple(image_features.shape))
print("same result:", torch.allclose(image_features_manual, image_features, atol=1e-4))

**Q2.4** The tokens have dimension 768 inside the ViT, but the image feature has dimension 512. What is `vit.proj` for?
(Look back at the text tower in Q1.1.)

## 3. Inside multi-head attention

Each residual block applies multi-head self-attention followed by an MLP. Let us look at the attention of the first block.

In [ ]:
print(attn0)
print({n: tuple(p.shape) for n, p in attn0.named_parameters()})

The function below computes multi-head self-attention from the raw weights `in_proj_weight`, `in_proj_bias` and
`out_proj`, following the recipe:

1. `qkv = x W_in^T + b_in`, then reshape to `[B, N, 3, H, d]` and move the axes to get `q, k, v` of shape `[B, H, N, d]`;
2. `attn = softmax(q k^T / sqrt(d))` along the last axis, shape `[B, H, N, N]`;
3. `out = attn v`, shape `[B, H, N, d]`, then merge the heads back to `[B, N, H*d]`;
4. apply the output projection `out_proj`.

It returns both the output and the attention weights; the check compares its output with the module's own forward pass.

In [ ]:
def multi_head_attention(x, attn_module):
    '''x: [B, N, C]. Returns (out [B, N, C], attn [B, H, N, N]).'''
    B, N, C = x.shape
    H, d = attn_module.num_heads, attn_module.head_dim

    qkv = F.linear(x, attn_module.in_proj_weight, attn_module.in_proj_bias)   # [B, N, 3C]
    qkv = qkv.reshape(B, N, 3, H, d).permute(2, 0, 3, 1, 4)                 # [3, B, H, N, d]
    q, k, v = qkv.unbind(0)                                                  # each [B, H, N, d]

    attn = (q @ k.transpose(-2, -1)) / math.sqrt(d)                          # [B, H, N, N]
    attn = attn.softmax(dim=-1)

    out = attn @ v                                                           # [B, H, N, d]
    out = out.transpose(1, 2).reshape(B, N, C)                               # [B, N, H*d]

    out = attn_module.out_proj(out)
    return out, attn

x_test = torch.randn(1, 197, width, device=device)
with torch.no_grad():
    out_manual, attn_manual = multi_head_attention(x_test, attn0)
    out_module = attn0(x_test, method="direct")
print("attention weights:", tuple(attn_manual.shape), "| rows sum to", attn_manual[0, 0, 0].sum().item())
print("same result:", torch.allclose(out_manual, out_module, atol=1e-4))

**Q3.1** The attention weights have shape `[1, 12, 197, 197]`. What does entry `attn[0, h, i, j]` represent? What does
row `i = 0` correspond to, and why does every row sum to 1?

### 3.1 Visualising the attention of the [CLS] token

We register a *forward hook* on the `attn_map` sub-module of a block: it is an identity layer placed in the code exactly
where the attention weights are computed, so the hook receives the `[1, 12, 197, 197]` tensor for free.

In [ ]:
def get_cls_attention(img, layer, num_register_tokens=0):
    '''Returns the attention weights [H, N, N] of block `layer` and the block's output tokens [N, C].'''
    store = {}
    block = vit.transformer.resblocks[layer]
    h1 = block.attn.attn_map.register_forward_hook(lambda m, i, o: store.__setitem__("attn", o))
    h2 = block.register_forward_hook(lambda m, i, o: store.__setitem__("tokens", o))
    with torch.no_grad():
        model.encode_image(img, num_register_tokens=num_register_tokens)
    h1.remove(); h2.remove()
    return store["attn"][0].float().cpu(), store["tokens"][0].float().cpu()

def show_heads(attn, title, n_patches=196):
    '''Grid of the [CLS] attention map of each head (row 0, patch columns only).'''
    maps = attn[:, 0, 1:1 + n_patches]                       # [H, 196]
    side = int(math.sqrt(n_patches))
    fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
    vmin, vmax = maps.min(), maps.max()
    for h, ax in enumerate(axes.flat):
        ax.imshow(maps[h].reshape(side, side), vmin=vmin, vmax=vmax); ax.set_title(f"head {h}"); ax.axis("off")
    fig.suptitle(title, fontsize=16); plt.tight_layout(); plt.show()

attn_first, _ = get_cls_attention(img, layer=0)
attn_last, tokens_last = get_cls_attention(img, layer=-1)
show_heads(attn_first, "[CLS] attention, first block")
show_heads(attn_last, "[CLS] attention, last block")

**Q3.2** Compare the heads of the first block with those of the last block. Do the heads of a same block look at the
same patches? What is special about the last block?

In [ ]:
mean_map = attn_last[:, 0, 1:].mean(0).reshape(14, 14)     # average over heads
token_norms = tokens_last[1:].norm(dim=-1).reshape(14, 14)  # L2 norm of each patch token after the last block

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(im); axes[0].set_title("image")
i0 = axes[1].imshow(mean_map); axes[1].set_title("mean [CLS] attention, last block"); plt.colorbar(i0, ax=axes[1])
i1 = axes[2].imshow(token_norms); axes[2].set_title("norm of the patch tokens"); plt.colorbar(i1, ax=axes[2])
for ax in axes: ax.axis("off")
plt.show()

**Q3.3** Is the mean [CLS] attention map a good "saliency map" of the shark? Compare it with the map of the token norms.
What do you conclude about the patches that receive most of the attention?

## 4. CLIP: zero-shot classification with text

CLIP was trained on 400M (image, caption) pairs to bring the embedding of an image and of its caption close together,
and to push apart the embeddings of non-matching pairs (a *contrastive* objective). As a result, we can classify an
image with **any list of class names**, without training: we embed the candidate captions and pick the closest one.

In [ ]:
labels = ["a car", "a shark", "a bird", "a ball"]

with torch.no_grad():
    image_features = model.encode_image(img)                          # [1, 512]
    text_features = model.encode_text(model.tokenize(labels).to(device))   # [4, 512]

image_features = image_features / image_features.norm(dim=-1, keepdim=True)
text_features = text_features / text_features.norm(dim=-1, keepdim=True)

similarity = image_features @ text_features.T                        # [1, 4] cosine similarities
logit_scale = clip.logit_scale.exp().item()
probs = (logit_scale * similarity).softmax(dim=-1)[0]

for label, sim, p in zip(labels, similarity[0].tolist(), probs.tolist()):
    print(f"{label:12s}  cosine = {sim:.3f}   prob = {p:.3f}")
print("logit_scale =", round(logit_scale, 1))

**Q4.1** Why are the image and text features normalised before the dot product? What does `similarity` measure?

**Q4.2** The raw cosine similarities are all close to each other (typically between 0.1 and 0.3), yet the
probabilities are very peaked. What is `logit_scale` and why is it needed before the softmax?

### 4.1 Prompt engineering

CLIP was trained on natural captions, so the *wording* of the class names matters. The function below embeds each
class name inside a template such as `"a photo of a {}."` (averaging the embeddings when several templates are given).
We compare the probabilities obtained with the bare names, with one template, and with the average of four templates,
on two harder label sets: shark species, and the type of scene.

In [ ]:
templates = ["a photo of a {}.", "a picture of a {}.", "a close-up photo of a {}.", "a {} in the wild."]
label_sets = {"species": ["great white shark", "tiger shark", "hammerhead shark", "dolphin"],
              "scene": ["aquarium", "ocean", "beach", "dune"]}

def zero_shot_probs(image_features, class_names, templates):
    '''image_features: normalised [1, 512]. Returns a [len(class_names)] tensor of probabilities.'''
    with torch.no_grad():
        class_embeddings = []
        for name in class_names:
            texts = [t.format(name) for t in templates]
            emb = model.encode_text(model.tokenize(texts).to(device))     # [T, 512]
            emb = emb / emb.norm(dim=-1, keepdim=True)
            emb = emb.mean(0)
            class_embeddings.append(emb / emb.norm())
        class_embeddings = torch.stack(class_embeddings)                 # [num_classes, 512]
        probs = (logit_scale * image_features @ class_embeddings.T).softmax(dim=-1)[0]
    return probs

for set_name, class_names in label_sets.items():
    print(f"--- {set_name}")
    for name, tpls in [("bare names", ["{}"]), ("one template", [templates[0]]), ("4 templates", templates)]:
        p = zero_shot_probs(image_features, class_names, tpls)
        print(f"{name:13s} " + "  ".join(f"{c}={v:.2f}" for c, v in zip(class_names, p.tolist())))

**Q4.3** Does the template always sharpen the prediction? Why do the CLIP authors nevertheless recommend prompt
templates (and ensembling several of them) for zero-shot classification?

### 4.2 An image-text similarity matrix

In [ ]:
# The image is resized to 224x224 *before* CLIP's preprocessor (whose CenterCrop would otherwise cut the sides off),
# so that the model sees the whole picture and the attention maps can be drawn over it.
im2 = Image.open("images/bird_and_shark.jpg").convert("RGB").resize((224, 224))
img2 = model.preprocessor(im2).unsqueeze(0).to(device)
captions = ["a shark swimming underwater", "a bird standing on a log", "a bird and a shark",
            "a photo of the sea", "a red sports car"]

with torch.no_grad():
    feats = model.encode_image(torch.cat([img, img2]))
    feats = feats / feats.norm(dim=-1, keepdim=True)
    tfeats = model.encode_text(model.tokenize(captions).to(device))
    tfeats = tfeats / tfeats.norm(dim=-1, keepdim=True)
sim = (feats @ tfeats.T).cpu()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [1, 1, 2.2]})
axes[0].imshow(im); axes[0].set_title("image 0"); axes[0].axis("off")
axes[1].imshow(im2); axes[1].set_title("image 1"); axes[1].axis("off")
axes[2].imshow(sim, cmap="viridis"); axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(["image 0", "image 1"])
axes[2].set_xticks(range(len(captions))); axes[2].set_xticklabels(captions, rotation=30, ha="right")
for i in range(2):
    for j in range(len(captions)):
        axes[2].text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="w")
axes[2].set_title("cosine similarity"); plt.tight_layout(); plt.show()

**Q4.4** Comment on the matrix: which caption wins for each image, and are the "wrong" captions equally far? The text
tower is also a Transformer: which token of the caption do you think is read out as the sentence embedding, and how is
that analogous to the [CLS] token of the ViT? (Hint: `model.tokenize(captions)` returns padded sequences that start
with a start-of-text token and end with an end-of-text token, and the text transformer uses a causal mask.)

## 5. What does CLIP look at?

### 5.1 Test-time registers

In Q3.3 we saw that a few background patches hijack the attention of the [CLS] token. Darcet et al. proposed to
**retrain** ViTs with extra *register* tokens that serve as scratch space, so that the patches are left alone. Jiang,
Dravid et al. showed that the neurons responsible for creating these outlier tokens can be identified, and that their
activation can be *moved* at test time into an extra token, **without any retraining**: this is the
`num_register_tokens` argument of `encode_image`. The register is appended as token 198.

In [ ]:
attn_reg, tokens_reg = get_cls_attention(img, layer=-1, num_register_tokens=1)
print("attention with one register:", tuple(attn_reg.shape))

mean_map_reg = attn_reg[:, 0, 1:-1].mean(0).reshape(14, 14)          # drop [CLS] (0) and the register (last)
norms_reg = tokens_reg[1:-1].norm(dim=-1).reshape(14, 14)

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
axes[0, 0].imshow(mean_map); axes[0, 0].set_title("mean [CLS] attention, no register")
axes[0, 1].imshow(mean_map_reg); axes[0, 1].set_title("mean [CLS] attention, 1 register")
axes[1, 0].imshow(token_norms); axes[1, 0].set_title("token norms, no register")
axes[1, 1].imshow(norms_reg); axes[1, 1].set_title("token norms, 1 register")
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()
print("attention received by the register token (mean over heads):", attn_reg[:, 0, -1].mean().item())

**Q5.1** What changed in the attention map and in the token norms once a register is added? Where did the "scratch
space" go?

In [ ]:
def overlay(image, map14, title, ax=None, alpha=0.5):
    '''Upsample a 14x14 map to the image size and draw it over the image.'''
    up = np.array(Image.fromarray(np.asarray(map14, dtype=np.float32)).resize(image.size, Image.BILINEAR))
    ax = ax or plt.gca()
    ax.imshow(image); ax.imshow(up, cmap="jet", alpha=alpha); ax.set_title(title); ax.axis("off")

attn2, _ = get_cls_attention(img2, layer=-1, num_register_tokens=1)
map2 = attn2[:, 0, 1:-1].mean(0).reshape(14, 14)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
overlay(im, mean_map_reg.numpy(), "[CLS] attention (1 register)", axes[0])
overlay(im2, map2.numpy(), "[CLS] attention (1 register)", axes[1])
plt.tight_layout(); plt.show()

### 5.2 Text-conditioned attention maps

The [CLS] attention tells us what the image encoder considers important, independently of any text. With CLIP we can
ask a sharper question: **which patches make the image similar to a given caption?**

We use a Grad-CAM-like recipe on the last attention layer:
1. compute the similarity score `s = <image_features, text_features>` for the caption;
2. back-propagate `s` to the attention weights `A` of the last block (`[1, 12, 197+1, 197+1]`);
3. weight the attention by its gradient, average over heads and keep the [CLS] row: `cam = mean_h( dS/dA * A )[0, patches]`, clipped at 0.

Patches whose attention *increases* the score light up.

In [ ]:
def text_conditioned_map(img, caption, num_register_tokens=1):
    '''Grad-CAM on the [CLS] attention of the last block, for the similarity with `caption`. Returns a 14x14 array.'''
    with torch.no_grad():
        t = model.encode_text(model.tokenize([caption]).to(device))
        t = t / t.norm(dim=-1, keepdim=True)

    store = {}
    def hook(m, i, o):
        o.retain_grad(); store["attn"] = o
    h = vit.transformer.resblocks[-1].attn.attn_map.register_forward_hook(hook)
    f = model.encode_image(img, num_register_tokens=num_register_tokens)
    f = f / f.norm(dim=-1, keepdim=True)
    score = (f * t).sum()
    score.backward()
    h.remove()

    A, G = store["attn"], store["attn"].grad                      # [1, H, N, N]
    cam = (G * A).mean(dim=1)[0, 0]                               # [N]: [CLS] row, averaged over heads
    n_extra = num_register_tokens
    cam = cam[1:len(cam) - n_extra] if n_extra else cam[1:]       # patches only
    return cam.clamp(min=0).detach().float().cpu().numpy().reshape(14, 14)

captions = ["a shark", "a bird", "the grass", "a bird standing on a tree log"]
fig, axes = plt.subplots(1, len(captions) + 1, figsize=(4 * (len(captions) + 1), 4))
axes[0].imshow(im2); axes[0].set_title("image"); axes[0].axis("off")
for cap, ax in zip(captions, axes[1:]):
    overlay(im2, text_conditioned_map(img2, cap), f'"{cap}"', ax)
plt.tight_layout(); plt.show()

**Q5.2** The vision tower never sees the caption, yet the map changes with the text. Explain where the text enters the
computation.

**Q5.3** CLIP was trained only on (image, caption) pairs, without any bounding box or segmentation mask. In light of
the maps above, comment on the sentence *"a ViT classifier is an implicit segmenter"*. What are the limits of these maps
(resolution, quality, what they can and cannot localise)?

### 5.3 Your turn

A busier picture: a street scene with several people, a dog, a bicycle, cars, shop signs... The image is much wider
than it is tall, so we feed the model a version squashed to 224x224 (as before, so that nothing is cropped) but draw the
maps over the image at its natural aspect ratio: `overlay` upsamples the 14x14 map to whatever size the displayed image has.

In [ ]:
im3 = Image.open("images/scene.jpg").convert("RGB")
im3_display = im3.resize((im3.width * 448 // im3.height, 448))                # for display only (keeps the aspect ratio)
img3 = model.preprocessor(im3.resize((224, 224))).unsqueeze(0).to(device)   # what the model sees

plt.figure(figsize=(12, 6)); plt.imshow(im3_display); plt.axis("off"); plt.show()

Write your own captions in the cell below and look at where the model "looks" for each of them. Some ideas: an object
(`"a dog"`, `"a bicycle"`, `"a cup of coffee"`), a colour or a material (`"red"`, `"cobblestones"`), text in the image
(`"the word CAFE"`), an action (`"someone riding a bike"`), a caption unrelated to the picture (`"a snowy mountain"`).

In [ ]:
my_captions = ["a dog", "a woman drinking coffee", "a bicycle", "a snowy mountain"]   # <-- change these
n = len(my_captions)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(14, 4 * ((n + 1) // 2)))
axes = np.atleast_1d(axes).ravel()
for cap, ax in zip(my_captions, axes):
    overlay(im3_display, text_conditioned_map(img3, cap), f'"{cap}"', ax)
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

**Q5.4** Which of your captions are well localised, which are not? What happens with the caption that does not match
the image at all? Relate your observations to the limits discussed in Q5.3.

## Wrap-up

- A ViT is a Transformer encoder over a sequence of **patch tokens** plus a **[CLS] token**, with **learned positional
  embeddings**; the patch embedding is a strided convolution, i.e. one linear layer shared by all patches.
- **Multi-head attention** computes Q, K, V with one fused linear layer, splits them into heads of dimension 64 and
  mixes the heads back with an output projection. The attention weights `[H, N, N]` can be read directly.
- The [CLS] attention of a plain ViT is polluted by **high-norm outlier tokens**; **registers** (trained or added at
  test time) move this scratch space out of the image.
- **CLIP** puts images and captions in a **shared space** with cosine similarity and a learned temperature, which enables
  zero-shot classification, prompt engineering, and **text-conditioned** localisation maps.